# Lab Week 1 · Data preprocessing

Run cells in order with **Shift + Enter**. Edit the code, inspect the output and write your explanations below each task.

This is synthetic customer data. Your notebook is saved in this browser, not submitted to your lecturer. Use the **Download** button in the notebook toolbar to keep your work. First launch needs an internet connection to load Python and packages.

Read the comments and output headings in each cell. Plots show missingness, the unusual income, scaling and one-hot encoding.

In [ ]:
# piplite loads Python packages inside this browser notebook.
import piplite
# Download the libraries needed for tables, plots and preprocessing.
# await waits until installation finishes before the next line runs.
await piplite.install(["pandas", "matplotlib", "scikit-learn"])
print("Python packages are ready.")

## Create the dataset
Eight customer observations, four numerical features and one category. Income is in rupees and spending score is on a 0–100 scale.

In [ ]:
# pandas works with tables; pd is its conventional short name.
import pandas as pd
# pyplot creates the graphs displayed below each code cell.
import matplotlib.pyplot as plt
# StandardScaler puts numerical features on a comparable scale.
from sklearn.preprocessing import StandardScaler

# Create a table of eight synthetic customers. Each row is one observation.
# None represents a missing measurement or category, not a missing label.
df = pd.DataFrame({
    "age": [20, 25, 32, 40, 44, 51, None, 28],
    "annual_income": [240000, 360000, 480000, None,
                      600000, 720000, 840000, 9999999],
    "spending_score": [30, 45, 65, 55, 70, None, 40, 90],
    "purchase_frequency": [2, 4, 6, 3, 7, 5, 4, 8],
    "preferred_channel": ["Shop", "Online", "Shop", "Online",
                          None, "Shop", "Online", "Shop"]
})
# Select the numerical input features; exclude the category column.
numeric = ["age", "annual_income", "spending_score",
           "purchase_frequency"]

## Task 1 · Inspect before changing anything

Display the dataset, check the column types and count missing values. Identify the observations and features. No labels or known outputs are supplied.

Record in your notebook: Record the row and column counts, and list the columns with missing values.

In [ ]:
# Inspect the original table before changing any values.
print(df)
# Check whether each column contains numbers or other values.
print(df.dtypes)
# isna marks missing cells; sum counts them in each column.
print("\nDisplay Missing values")
print(df.isna().sum())

# Each bar counts the missing cells in one feature. Zero means no blanks.
df.isna().sum().plot.barh(color="teal", figsize=(7, 3))
plt.title("Missing values before preprocessing")
plt.xlabel("Number of missing values")
plt.xticks(range(3))
plt.tight_layout()
plt.show()

**My explanation:**

Double-click this cell to write your explanation, then press Shift + Enter.

## Task 2 · Handle missing values

Keep the original dataset. Make a working copy, fill numerical blanks with the column median and replace the missing channel with “Unknown”. Compare this with dropping incomplete rows.

Record in your notebook: Explain why you chose to fill the blanks rather than drop the rows. State one limitation of estimating a missing value.

In [ ]:
# Work on a copy so the original measurements remain available.
clean = df.copy()
# Estimate each missing numerical value using that column's median.
# This fills a blank; it does not recover the true missing measurement.
clean[numeric] = clean[numeric].fillna(clean[numeric].median())
# Use a separate category for an unknown channel instead of guessing.
clean["preferred_channel"] = clean["preferred_channel"].fillna("Unknown")

print("\nMissing values AFTER filling")
print(clean.isna().sum())
print("\nRows if we dropped incomplete observations instead:", len(df.dropna()))
print("Rows retained by filling:", len(clean))

# A missingness map shows exactly which cells changed from blank to filled.
# Yellow = missing; purple = present. Both panels use the SAME colour scale.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, table, title in zip(axes, [df, clean], ["Before filling", "After filling"]):
    ax.imshow(table.isna(), cmap="viridis", vmin=0, vmax=1, aspect="auto")
    ax.set_title(title)
    ax.set_xticks(range(len(df.columns)))
    ax.set_xticklabels(df.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(range(1, len(df) + 1))
axes[0].set_ylabel("Customer observation")
plt.suptitle("Yellow = missing | Purple = present (filled or originally observed)")
plt.tight_layout()
plt.show()

**My explanation:**

Double-click this cell to write your explanation, then press Shift + Enter.

## Task 3 · Investigate an unusual value

Inspect annual income. The final value is much larger than the others. In this synthetic example we cannot verify its source, so keep it for now and note that it needs checking.

Record in your notebook: Describe how you would verify the income. Explain why an unusual value should not automatically be deleted.

In [ ]:
# Summary statistics and sorting help us locate unusually large incomes.
print("\nIncome summary")
print(clean["annual_income"].describe())
print("\nCustomers sorted by income")
print(clean.sort_values("annual_income"))

# These two panels plot the SAME data. The right panel only zooms the x-axis.
# Dividing by 1,000,000 makes the axis easier to read: income in Rs millions.
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax in axes:
    ax.scatter(clean["annual_income"] / 1_000_000, clean["spending_score"])
    ax.set_xlabel("Annual income (Rs millions)")
axes[0].set_title("All observations, including the unusual income")
axes[0].annotate("Check this income", (9.999999, 90), xytext=(3, 80),
                 arrowprops={"arrowstyle": "->"})
axes[1].set_xlim(0, 1)
axes[1].set_title("Zoom: Rs 0–1 million (data unchanged)")
axes[0].set_ylabel("Spending score")
plt.tight_layout()
plt.show()
# The unusual observation is still in clean. Verify its source before deciding
# whether it is an error or a valid rare value; being unusual is not enough.

**My explanation:**

Double-click this cell to write your explanation, then press Shift + Enter.

## Task 4 · Put numerical features on a comparable scale

Standardise the four numerical features. Keep the result separate so you can compare it with the original units. Notice how the unusually large income affects the scaled values.

Record in your notebook: Compare the income and spending columns before and after scaling. Explain why scaling can matter when comparing observations.

In [ ]:
# For each numerical feature, subtract its mean and divide by its standard
# deviation. Keep the original clean table unchanged for comparison.
# For later predictive tasks, fit preprocessing settings on training data only.
scaled = pd.DataFrame(
    StandardScaler().fit_transform(clean[numeric]),
    columns=numeric, index=clean.index
)
print("\nBEFORE scaling: original units")
print(clean[numeric].head())
print("\nAFTER scaling: standardised values")
print(scaled.round(2))  # round only changes the displayed precision.

# Box plots summarise each feature's spread; isolated markers show extreme values.
# Original columns use different units. Standardised columns use standard deviations.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, table, title in zip(axes, [clean[numeric], scaled],
                            ["Before: mixed original units", "After: standardised values"]):
    ax.boxplot([table[column] for column in numeric])
    ax.set_xticks(range(1, len(numeric) + 1))
    ax.set_xticklabels(["Age", "Income", "Spending", "Purchases"], rotation=30)
    ax.set_title(title)
axes[0].set_ylabel("Original feature units (different for each column)")
axes[1].set_ylabel("Standard deviations from the feature mean")
plt.tight_layout()
plt.show()
# Scaling changes numerical units; it does not remove the unusual income
# or guarantee useful groups. Each panel has its own vertical axis.

**My explanation:**

Double-click this cell to write your explanation, then press Shift + Enter.

## Task 5 · Convert categories into numerical features

One-hot encode the preferred channel and combine it with the scaled numerical features. Keep all category columns for this exercise.

Record in your notebook: Show the columns for Online, Shop and Unknown. Explain what a 1 and a 0 mean, and why category codes 1, 2 and 3 could be misleading.

In [ ]:
# One-hot encoding creates one 0/1 input feature for each category.
# This avoids inventing a numerical order for Online, Shop and Unknown.
channels = pd.get_dummies(
    clean["preferred_channel"], prefix="channel", dtype=int
)
# Join the scaled numbers and category indicators using the same row indices.
prepared = pd.concat([scaled, channels], axis=1)
print("\nOne-hot category columns")
print(channels)
print("\nFinal prepared feature table")
print(prepared.round(2))
print("\nMissing values remaining:", prepared.isna().sum().sum())

# Each row is a customer. A bright square (1) marks their channel;
# a pale square (0) means that channel is absent for that observation.
fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(channels, cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(channels.columns)))
ax.set_xticklabels(channels.columns, rotation=20, ha="right")
ax.set_yticks(range(len(channels)))
ax.set_yticklabels(range(1, len(channels) + 1))
for row in range(len(channels)):
    for column in range(len(channels.columns)):
        value = channels.iloc[row, column]
        ax.text(column, row, str(value), ha="center", va="center",
                color="white" if value == 1 else "black")
ax.set_title("One-hot encoding: 1 = present, 0 = absent")
ax.set_ylabel("Customer observation")
plt.tight_layout()
plt.show()

**My explanation:**

Double-click this cell to write your explanation, then press Shift + Enter.

## Task 6 · Explain your prepared dataset

Compare df with prepared. Check that every row is still present and that no missing values remain. Write a short explanation of your choices and one thing you would investigate next.

Record in your notebook: Your notebook should contain the original data, inspection results, preprocessing steps, a plot, the prepared table and brief explanations.

In [ ]:
# shape reports (observations, features). Encoding adds columns, not customers.
print("\nOriginal dataset shape:", df.shape)
print("Prepared dataset shape:", prepared.shape)
print("\nFinal checks")
# assert stops the cell with an error if the condition is not true.
assert len(prepared) == len(df), "An observation was lost."
assert not prepared.isna().any().any(), "Some feature values are still missing."
print("All observations retained and no missing values remain.")
# These checks confirm the table's structure, not that every decision was correct.
# Add your explanation of the filling, unusual-value, scaling and encoding choices.

**My explanation:**

Double-click this cell to write your explanation, then press Shift + Enter.

## Task 7 · Your turn: preprocess a new dataset

Apply the ideas from the guided customer example to these **synthetic student study habits**. There are ten observations, four numerical features and one categorical feature. No labels or known outputs are supplied. Keep this work separate from the guided example by using the `practice` table.

1. Inspect the observations, feature types, missing values and possible unusual values.
2. Keep the original data. Choose how to handle missing numerical and categorical values, and justify your choices.
3. Investigate any unusual observation. Explain what you would verify before keeping, correcting or excluding it; do not remove a value only because it is large.
4. Choose appropriate numerical scaling and encode the categorical feature without inventing an order.
5. Produce plots that show at least two preprocessing changes or decisions. Label the axes and explain what the plots show.
6. Present the prepared feature table. Check the retained rows and remaining missing values, and explain what changed.

**Write your own preprocessing code.** Add comments explaining your decisions and interpretations, rather than only describing the Python commands. Conclude with one limitation and explain how a preprocessing choice could affect which students appear similar in a later clustering task. Do not run clustering yet.

**Your final notebook should include:** the new data, your inspection, your own preprocessing steps, before-and-after evidence, the prepared table and your explanations. There is no worked solution for this task.

In [ ]:
# A new synthetic dataset: each row is one student observation.
# These are input features. No exam grades or other known outputs are supplied.
import pandas as pd

practice = pd.DataFrame({
    "study_hours_week": [5, 8, None, 12, 6, 15, 9, 7, 11, 4],
    "attendance_pct": [70, 85, 90, None, 65, 95, 80, 75, 88, 60],
    "online_sessions_week": [4, 6, 8, 10, 3, 12, 7, 1200, None, 2],
    "commute_minutes": [15, 30, 45, 20, None, 60, 25, 40, 35, 10],
    "study_location": ["Home", "Library", "Campus", "Home", "Library",
                       None, "Campus", "Home", "Library", "Campus"]
}, index=["S01", "S02", "S03", "S04", "S05",
          "S06", "S07", "S08", "S09", "S10"])

# IDs identify observations; do not treat them as numerical input features.
# Units: hours and sessions per week, attendance in %, commute in minutes.
print(practice)

In [ ]:
# Write your own inspection and preprocessing code here.
# Add cells as needed for plots, checks and explanations.
# Keep the original practice table unchanged.

**My decisions and interpretation:**

Explain your missing-value handling, unusual-value decision, scaling and encoding. What changed in your plots? What limitation remains? How might these choices affect similarity?